In [14]:
import pandas as pd

import torch

from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification

from transformers import BertModel
from transformers import AutoTokenizer
from datasets import load_dataset

from tqdm import tqdm

from sklearn.metrics import accuracy_score, classification_report
rubert = 'DeepPavlov/rubert-base-cased'
EPOCHS = 3

In [20]:
dataset_train = load_dataset('csv', data_files='qwen3_14b_labeling_clean_3211q.csv');
dataset_val = load_dataset('csv', data_files='gold_manual_anno.csv');
print(f'{dataset_train}\n\n{dataset_val}')

DatasetDict({
    train: Dataset({
        features: ['question', 'class', 'label'],
        num_rows: 3211
    })
})

DatasetDict({
    train: Dataset({
        features: ['question', 'final'],
        num_rows: 329
    })
})


In [6]:
tokenizer = AutoTokenizer.from_pretrained(rubert);

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [16]:
def preprocess_function(example):
    return tokenizer(example['question'], padding='max_length', truncation=True, max_length=128)

def collate_to_device(batch):
    keys = batch[0].keys()
    out = {}

    for k in keys: out[k] = torch.stack([item[k] for item in batch]).to(device)
    return out


In [28]:
train_tok = dataset_train.map(preprocess_function, batched=True);
val_tok = dataset_val.map(preprocess_function, batched=True);

train_tok = train_tok.rename_column('label', 'labels')
train_tok = train_tok.remove_columns(['question'])
val_tok = val_tok.rename_column('final', 'labels')
val_tok = val_tok.remove_columns(['question'])


train_tok.set_format('torch');
val_tok.set_format('torch');

In [31]:
print(f'{train_tok}\n\n{val_tok}')

DatasetDict({
    train: Dataset({
        features: ['class', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3211
    })
})

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 329
    })
})


In [32]:
train_loader = DataLoader(train_tok['train'], batch_size=16, collate_fn=collate_to_device)
val_loader = DataLoader(val_tok['train'], batch_size=16, collate_fn=collate_to_device)

In [33]:
model = BertForSequenceClassification.from_pretrained(rubert, num_labels=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [34]:
optimizer = AdamW(model.parameters(), lr=2e-5)

In [35]:
for epoch in range(EPOCHS):

    ####### TRAIN #######
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f'train epoch: {epoch+1} : ', ncols=100)

    for i, batch in enumerate(pbar):

        outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'],
            token_type_ids=batch['token_type_ids'], labels=batch['labels'])

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # update tqdm with %
        percent = (i + 1) / len(train_loader) * 100
        pbar.set_postfix({'progress': f'{percent:.1f}%'})

    avg_train_loss = total_loss / len(train_loader)
    print(f'train epoch: {epoch+1} : {avg_train_loss:.4f}')

    ####### VALIDATION #######
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            outputs = model(input_ids=batch['input_ids'],
                            attention_mask=batch['attention_mask'],
                            token_type_ids=batch['token_type_ids'])

            preds = outputs.logits.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(batch['labels'].cpu().tolist())

    accuracy = 100 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    print(f'epoch: {epoch+1} validation accuracy: {accuracy:.2f}%')

    print('\nClassification Report:')
    print(classification_report(all_labels, all_preds, digits=4))


train epoch: 1 : 100%|███████████████████████████| 201/201 [01:16<00:00,  2.61it/s, progress=100.0%]


train epoch: 1 : 0.4675
epoch: 1 validation accuracy: 79.94%

Classification Report:
              precision    recall  f1-score   support

           0     0.7238    0.9500    0.8216       160
           1     0.9328    0.6568    0.7708       169

    accuracy                         0.7994       329
   macro avg     0.8283    0.8034    0.7962       329
weighted avg     0.8311    0.7994    0.7955       329



train epoch: 2 : 100%|███████████████████████████| 201/201 [01:15<00:00,  2.65it/s, progress=100.0%]


train epoch: 2 : 0.3226
epoch: 2 validation accuracy: 85.11%

Classification Report:
              precision    recall  f1-score   support

           0     0.8000    0.9250    0.8580       160
           1     0.9167    0.7811    0.8435       169

    accuracy                         0.8511       329
   macro avg     0.8583    0.8530    0.8507       329
weighted avg     0.8599    0.8511    0.8505       329



train epoch: 3 : 100%|███████████████████████████| 201/201 [01:15<00:00,  2.66it/s, progress=100.0%]


train epoch: 3 : 0.1894
epoch: 3 validation accuracy: 84.19%

Classification Report:
              precision    recall  f1-score   support

           0     0.8375    0.8375    0.8375       160
           1     0.8462    0.8462    0.8462       169

    accuracy                         0.8419       329
   macro avg     0.8418    0.8418    0.8418       329
weighted avg     0.8419    0.8419    0.8419       329

